
# Wordle letter explorer

Which letters does an optimal strategy reach for **first**? This notebook ports the
hangman tier-list generator (the last section of
`hangman/math/notebooks/letter_analysis.ipynb`, "Greedily grow the set to minimise
outside-before-set pairs") to Wordle, over the committed optimal trees.

**The Wordle adaptation.** In hangman a guess *is* a letter, so a game hands you a
strict order over letters. In Wordle a guess is a **word**, so up to five letters
first appear at the same moment. A letter's *appearance time* on a path is therefore
the index of the first guess WORD containing it, and letters that enter on the same
guess are **tied — neither is before the other, so they are never inverted**.

Everything else follows hangman: the never-guessed penalty, uniform weighting, the
greedy accumulation, and `top2_diff` as the tier-boundary hint.

> **The objective these trees optimise is CONSTRAINED.** Each tree minimises the
> TOTAL guesses over the 2315 answers *subject to resolving every answer within 6*.
> That is not the same as minimum expected guesses, and any ranking derived here
> inherits that constraint.

In [1]:

from pathlib import Path
import sys

def find_math_dir():
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (c / 'wordle.py').exists():
            return c
    raise FileNotFoundError('Could not find wordle.py. Open from the repo root, math/, or math/notebooks/.')

math_dir = find_math_dir()
for p in (math_dir, math_dir / 'tools'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import gzip
import re
import numpy as np
import pandas as pd
from IPython.display import display
import paths                      # the canonical solver locations (math/tools/paths.py)

pd.set_option('display.max_rows', 200)   # show up to 200 rows before pandas truncates to a head/tail preview

print(f'imported paths from {math_dir}')
print(f'solver = {paths.SOLVER!r}')
print(f'trees  = {paths.TREES}')

imported paths from /Users/patrickliscio/Documents/Ballpark Figures/ballpark-figures/wordle/math
solver = 'optturns'
trees  = /Users/patrickliscio/Documents/Ballpark Figures/ballpark-figures/wordle/math/data/shared/optturns/trees


## Defaults

`NUM_TREES` — how many optimal trees to pool. The openers are ranked by their **total
guesses over the 2315 answers** (equivalently by `expected`, the same ordering) from the
committed `opener_totals.csv`, ties broken alphabetically, and the best `NUM_TREES` are
taken. `NUM_TREES = 1` is **SALET alone** (total 7920). Every pooled path carries the same
weight, so larger values average the letter order across several equally good strategies.

`MAX_REPEAT` — how many copies of a letter get their **own token**. `1` is letters only.
`2` adds "two Es" (`ee`), which first appears on the guess where you first play a word
containing two Es; `3` adds "three Es" (`eee`) as well. So you can read off whether the
strategy reaches a second E before it reaches a Z.

> **3 is the ceiling in this data** — no word in either list repeats a letter four times
> (`guess.txt`: 4520 words hit 2, 130 hit 3; `mystery.txt`: 729 and 20). Setting
> `MAX_REPEAT = 4` is allowed but only adds 26 tokens that can never appear, and the
> never-guessed penalty will bury them all at the bottom.

> Cost scales as O(tokens² × paths), so each step up in `MAX_REPEAT` multiplies the work:
> 26 tokens → 351 evaluations, 52 → 1378 (~3.9×), 78 → 3081 (~8.8×). Start small.

In [2]:
NUM_TREES  = 1    # <-- how many optimal trees to pool (top N openers by expected guesses; 1 = SALET)
MAX_REPEAT = 3    # <-- 1 = letters only; 2 also ranks "two Es"; 3 also ranks "three Es"
MIN_GAP    = 0.5  # <-- greedy printout: blank line when top2_diff exceeds this (a tier-boundary hint)

_repeats = {1: 'letters only', 2: 'letters + doubles', 3: 'letters + doubles + triples'}
print(f'pooling the top {NUM_TREES} tree(s); '
      f'{_repeats.get(MAX_REPEAT, f"repeats up to {MAX_REPEAT}")}; '
      f'blank line in the greedy log at top2_diff > {MIN_GAP}')

pooling the top 1 tree(s); letters + doubles + triples; blank line in the greedy log at top2_diff > 0.5


## Loading the paths

Each tree file is one line per answer:

    <answer> <nguesses> <g1> <p1> <g2> <p2> ... <gN>

so **every line already is a path** — no tree walk needed. The guesses are `parts[2::2]`
(the same slicing `tools/checktrees.py` and `tools/depths.py` use), and the final guess
is always the answer itself, so the answer's own letters always appear.

**Tokens.** With `MAX_REPEAT = 1` a token is just a letter, and its appearance time is
the index of the first guess word containing it. Raising it gives `MAX_REPEAT` tokens per
letter:

- `e` — first guess word containing **at least one** E;
- `ee` — first guess word containing **at least two** Es;
- `eee` — first guess word containing **at least three** Es.

All are read off the guess you *typed*, regardless of colour, exactly as before. A word
with two Es makes `e` and `ee` appear **on the same guess**, so they are tied — a repeat
token can never rank ahead of its own shorter form, but it can rank behind other letters,
which is the whole point of the option.

Letters (or doubles) sharing a rank are tied and never inverted. `occ` is the token's
number of occurrences **in the answer**, which is what "revealed" means for
`set_before_outside`; see the note there for why a double token carries `occ = 0`.

The summed guess counts are checked against each opener's CSV total — the cheapest
end-to-end gate that the tree matches its reported value.

In [3]:
BIG = 1 << 30          # padding rank: never earlier than anything real
MAX_PREFIX = 6         # report letter coverage after guesses 1..6 (the normal-mode cap)


def n_tokens(max_repeat=None):
    """Size of the token universe: 26 letters per repeat level."""
    return 26 * (MAX_REPEAT if max_repeat is None else max_repeat)


def token_label(t):
    """Display label for a token index: 0..25 -> 'a'..'z', 26..51 -> 'aa'..'zz', etc."""
    return chr(97 + t % 26) * (t // 26 + 1)


def top_openers(num_trees):
    """The best `num_trees` openers by total guesses over the 2315 answers.

    Sourced from `paths.load_totals()` (the committed sweep CSV) -- never re-derived.
    `total` and `expected` induce the same ordering (expected = total / 2315); ties are
    broken alphabetically, so a strict top-N is taken."""
    totals = paths.load_totals()
    ranked = sorted(totals.items(), key=lambda kv: (kv[1], kv[0]))
    if num_trees > len(ranked):
        raise ValueError(f'only {len(ranked)} openers solved, asked for {num_trees}')
    return ranked[:num_trees]


def read_tree(opener):
    """Every path of one opener's optimal tree as (answer, [guess, ...])."""
    p = Path(paths.TREES) / f'{opener}.txt.gz'
    if not p.exists():
        raise FileNotFoundError(f'no committed tree for {opener!r} at {p}')
    out = []
    with gzip.open(p, 'rt') as f:
        for line in f:
            parts = line.split()
            out.append((parts[0], parts[2::2]))
    return out


def collect_paths(num_trees=None, max_repeat=None, verbose=True):
    """Pool the top `num_trees` trees' paths into padded arrays (built ONCE -- a path's
    token order does not depend on the candidate set).

    A token's appearance time is the index of the first guess WORD containing it: at least
    one copy of the letter for `e`, at least two for `ee`, at least three for `eee`, up to
    `max_repeat`. Tokens entering on the same guess are TIED.

    Returns a dict:
      order (P, M) int   token indices in first-appearance order, padded with `nt`
      rank  (P, M) int   the index of the guess WORD the token first appeared in
      block (P, M) int   column index where this column's rank-block starts (the tie group)
      occ   (P, M) int   what this token REVEALS about the answer: for a single letter, its
                         number of occurrences; for a REPEAT token, 1 when the answer really
                         does hold that many copies, else 0 -- knowing "there are two Es" is
                         its own unit of knowledge
      valid (P, M) bool  real (non-padding) columns
      nt    int          size of the token universe (26 per repeat level)
      n     int          number of pooled paths
      openers            [(opener, total), ...] actually pooled
      prefix_occ   (6, 26) row k-1: letters TYPED in the first k guesses, over every path
      prefix_paths (6, 26) row k-1: paths whose first k guesses contain the letter at all
                           (both cumulative in k, so row 5 covers guesses 1-6)
    """
    if num_trees is None:
        num_trees = NUM_TREES
    if max_repeat is None:
        max_repeat = MAX_REPEAT
    NT = n_tokens(max_repeat)
    openers = top_openers(num_trees)

    recs = []
    seen_max = 0                                    # deepest repeat actually played
    pre_occ = np.zeros((MAX_PREFIX, 26), dtype=np.int64)   # letters typed AT guess k-1
    pre_hit = np.zeros((MAX_PREFIX, 26), dtype=np.int64)   # letter FIRST appears at guess k-1
    for opener, total in openers:
        tree = read_tree(opener)
        assert len(tree) == paths.N_ANSWERS, f'{opener}: {len(tree)} paths, expected {paths.N_ANSWERS}'
        assert sum(len(g) for _a, g in tree) == total, f'{opener}: tree sum != CSV total {total}'
        for answer, guesses in tree:
            first = {}
            for gi, g in enumerate(guesses):
                counts = {}
                for ch in g:
                    counts[ch] = counts.get(ch, 0) + 1
                for ch, k in counts.items():
                    seen_max = max(seen_max, k)
                    t = ord(ch) - 97
                    if gi < MAX_PREFIX:
                        pre_occ[gi, t] += k                      # every copy typed at guess gi
                    for r in range(1, min(k, max_repeat) + 1):   # 'e', then 'ee', then 'eee'
                        first.setdefault((r - 1) * 26 + t, gi)
            for t, r in first.items():                           # where each LETTER debuts
                if t < 26 and r < MAX_PREFIX:
                    pre_hit[r, t] += 1
            toks = sorted(first, key=lambda t: (first[t], t))    # ties keep a stable order
            recs.append((toks,
                         [first[t] for t in toks],
                         [answer.count(token_label(t)) if t < 26
                          else int(answer.count(token_label(t)[0]) >= t // 26 + 1)
                          for t in toks]))

    P = len(recs)
    M = max(len(o) for o, _r, _c in recs)
    order = np.full((P, M), NT, dtype=np.int64)                 # NT = the padding sentinel
    rank = np.full((P, M), BIG, dtype=np.int64)
    occ = np.zeros((P, M), dtype=np.int64)
    vlen = np.zeros(P, dtype=np.int64)
    for i, (o, r, c) in enumerate(recs):
        k = len(o)
        order[i, :k], rank[i, :k], occ[i, :k], vlen[i] = o, r, c, k

    col = np.arange(M)
    valid = col[None, :] < vlen[:, None]
    # block[i, k] = first column of the tie group (same rank) containing column k.
    # rank is non-decreasing along a row, so a running max of the group starts works.
    is_new = np.ones((P, M), dtype=bool)
    is_new[:, 1:] = rank[:, 1:] != rank[:, :-1]
    block = np.maximum.accumulate(np.where(is_new, col[None, :], 0), axis=1)

    if verbose:
        names = ', '.join(f'{w} ({t})' for w, t in openers[:6])
        more = '' if len(openers) <= 6 else f', ... (+{len(openers) - 6} more)'
        print(f'pooled {P:,} paths from {len(openers)} tree(s): {names}{more}')
        print(f'{NT} tokens (26 x repeats up to {max_repeat}); '
              f'deepest repeat actually guessed anywhere: {seen_max}; '
              f'per path: min {vlen.min()}, mean {vlen.mean():.2f}, max {vlen.max()}')
        if max_repeat > seen_max:
            print(f'  NOTE: no guess ever repeats a letter more than {seen_max} times, so '
                  f'{26 * (max_repeat - seen_max)} tokens can never appear and will rank last.')
    return {'order': order, 'rank': rank, 'block': block, 'occ': occ, 'valid': valid,
            'nt': NT, 'n': P, 'openers': openers,
            'prefix_occ': np.cumsum(pre_occ, axis=0),            # cumulative over guesses
            'prefix_paths': np.cumsum(pre_hit, axis=0)}


D = collect_paths()

pooled 2,315 paths from 1 tree(s): salet (7920)
78 tokens (26 x repeats up to 3); deepest repeat actually guessed anywhere: 3; per path: min 5, mean 11.94, max 21


## The two metrics

Given a set of tokens `S`, over every pooled path:

- **`pairs_outside_before_set`** *(this is what the greedy minimises)* — for each set
  token α, the number of outside tokens (β∉S) that first appeared **strictly earlier**; a
  β entering on the **same guess** as α is tied and does **not** count. **And if α never
  appears on that path at all, every outside token that does appear counts against it** —
  the never-guessed penalty, so rare tokens are penalised rather than rewarded for their
  absence. Summed over α∈S. Doubles are heavily penalised by this rule, which is the
  intended behaviour: a second E you rarely get to play should not rank above a Z you do.
- **`set_before_outside`** *(reported alongside, does not steer the greedy)* — the number
  of set-token **occurrences in the answer** that are revealed before the first outside
  token is typed. A token entering on the same guess as that first outside token is tied,
  so it does not count as before.

> **A repeat token counts as its OWN unit of knowledge.** `ee` contributes **1** when the
> answer really does hold two or more Es, `eee` another **1** when it holds three — on top
> of whatever the single token `e` contributes. So the metric reads as *units of knowledge
> revealed*, not *letters of the word filled in*: above `MAX_REPEAT = 1` the whole-universe
> total exceeds 5 (it is 5, plus one per letter the answer doubles, plus one per letter it
> triples), where at `MAX_REPEAT = 1` it is exactly 5. Keep that in mind before comparing
> `set_before_outside` across different `MAX_REPEAT` — they are different quantities on
> different scales.

Both are averaged **uniformly over the pooled paths** (the Wordle sweep carries no
frequency weighting), so with `NUM_TREES = 1` that is a plain average over the 2315
answers.

In [4]:
def _token_mask(spec, D=None):
    """Boolean mask over the token universe, plus one padding slot at index `nt`.

    `spec` is a string of tokens. With a SEPARATOR (space or comma) each piece is a
    token: 'e' is one E, 'ee' is two Es, 'eee' is three. WITHOUT a separator every
    character is its own single-letter token, so 'aeiou' still means the five vowels."""
    D = D if D is not None else globals()['D']
    NT = D['nt']
    mask = np.zeros(NT + 1, dtype=bool)
    pieces = re.split(r'[\s,]+', str(spec).lower().strip()) if re.search(r'[\s,]', str(spec)) \
        else list(str(spec).lower())
    for piece in pieces:
        if not piece:
            continue
        if not piece.isalpha() or piece != piece[0] * len(piece):
            raise ValueError(f'{piece!r} is not a token (expected one letter repeated 1..n times)')
        if 26 * len(piece) > NT:
            raise ValueError(f'{piece!r} needs MAX_REPEAT >= {len(piece)}, but it is {NT // 26}')
        mask[(len(piece) - 1) * 26 + ord(piece[0]) - 97] = True
    return mask


def all_tokens(D=None):
    """The whole token universe as a spec string -- the set for which pairs must be 0."""
    D = D if D is not None else globals()['D']
    return ' '.join(token_label(t) for t in range(D['nt']))


def pairs_total(inS, D=None):
    """Total outside-before-set pairs over every pooled path (see the markdown above)."""
    D = D if D is not None else globals()['D']
    valid = D['valid']
    inS_seq = inS[D['order']] & valid
    out_seq = valid & ~inS_seq
    cum = np.cumsum(out_seq, axis=1) - out_seq              # outside tokens in EARLIER columns
    before = np.take_along_axis(cum, D['block'], axis=1)    # ... rewound to the tie group's start,
                                                            #     so same-guess tokens never count
    old = (inS_seq * before).sum(axis=1)                    # pairs for set tokens that DO appear
    n_out = out_seq.sum(axis=1)                             # outside tokens appearing on this path
    k_missing = int(inS[:D['nt']].sum()) - inS_seq.sum(axis=1)   # set tokens that never appear
    return float((old + k_missing * n_out).sum())


def before_total(inS, D=None):
    """Total set-token occurrences in the answer revealed before the first outside token."""
    D = D if D is not None else globals()['D']
    valid = D['valid']
    inS_seq = inS[D['order']] & valid
    out_seq = valid & ~inS_seq
    first_out = np.where(out_seq, D['rank'], BIG).min(axis=1)   # rank of the first outside token
    mask = inS_seq & (D['rank'] < first_out[:, None])           # strictly earlier -> ties excluded
    return float((D['occ'] * mask).sum())


def letter_set_stats(spec, D=None):
    """Both metrics for one explicit token set, averaged over the pooled paths."""
    D = D if D is not None else globals()['D']
    inS = _token_mask(spec, D)
    P = D['n']
    return {'set': ' '.join(token_label(t) for t in range(D['nt']) if inS[t]),
            'set_before_outside': before_total(inS, D) / P,
            'pairs_outside_before_set': pairs_total(inS, D) / P}


# Sanity: the whole token universe has no outside tokens, so pairs must be 0, and every one
# of the answer's five letter-occurrences is revealed before a (non-existent) outside token.
_demo = [all_tokens(), 'salet', 'aeiou']
if MAX_REPEAT >= 2:
    _demo += ['e ee']                   # E, and a second E as a token of its own
if MAX_REPEAT >= 3:
    _demo += ['e ee eee']
display(pd.DataFrame([letter_set_stats(s) for s in _demo]))

,set,set_before_outside,pairs_outside_before_set
0,a b c d e f g h i j k l m n o p q r s t u v w ...,5.349028,0.000000
1,a e l s t,1.869978,0.000000
2,a e i o u,0.000000,17.834989
3,e ee,0.000000,10.366307
4,e ee eee,0.000000,21.161555


## Greedily grow the set to minimise outside-before-set pairs

Starting from the empty set, repeatedly add the outside token that **minimises
`pairs_outside_before_set`** (with the never-guessed rule, so rare tokens are not
favoured), until every token is in. Ties are broken by token index — singles
alphabetically, then doubles alphabetically.

Each step prints the added token, the resulting set, that set's `set_before_outside` and
`pairs_outside_before_set`, and **`top2_diff`** — the runner-up candidate's pairs minus
the chosen token's, i.e. how decisively the winner beat the next-best option (NaN at the
final step). A blank line marks `top2_diff > MIN_GAP`.

Because a path's token order does not depend on the set, the orders are collected once
(above) and each candidate set is scored by a vectorised pass over the padded arrays.

In [5]:
def greedy_min_pairs(D=None, min_gap=None):
    """Greedily add one token at a time -- each step the outside token that MINIMISES
    pairs_outside_before_set -- until the whole universe is in. Returns the df."""
    D = D if D is not None else globals()['D']
    if min_gap is None:
        min_gap = MIN_GAP
    NT, P = D['nt'], D['n']
    inS = np.zeros(NT + 1, dtype=bool)
    rows, chosen, remaining = [], [], list(range(NT))

    names = ', '.join(w for w, _t in D['openers'][:4])
    more = '' if len(D['openers']) <= 4 else f' (+{len(D["openers"]) - 4} more)'
    print(f'greedy min pairs_outside_before_set  |  {len(D["openers"])} tree(s): {names}{more}'
          f'  |  {P:,} paths, {NT} tokens, uniform')

    for step in range(NT):
        scored = []
        for x in remaining:                          # pairs total for each outside candidate
            inS[x] = True
            scored.append((pairs_total(inS, D), x))
            inS[x] = False
        scored.sort()                                # (pairs, token) asc -> ties by token index
        best_tot, best_x = scored[0]
        second_tot = scored[1][0] if len(scored) > 1 else None

        inS[best_x] = True
        remaining.remove(best_x)
        chosen.append(best_x)

        p_avg = best_tot / P
        gap = (second_tot - best_tot) / P if second_tot is not None else np.nan
        b_avg = before_total(inS, D) / P
        label = token_label(best_x)
        setstr = ' '.join(sorted(token_label(c) for c in chosen))
        rows.append({'step': step + 1, 'added': label, 'set': setstr,
                     'set_before_outside': round(b_avg, 4),
                     'pairs_outside_before_set': round(p_avg, 4),
                     'top2_diff': round(gap, 4) if second_tot is not None else np.nan})
        print(f'  step {step + 1:2d}: +{label:<3}  '
              f'set_before_outside={b_avg:7.4f}   pairs={p_avg:7.4f}   top2_diff={gap:7.4f}')
        if second_tot is not None and gap > min_gap:
            print()
    return pd.DataFrame(rows)


sel = greedy_min_pairs()
display(sel)

greedy min pairs_outside_before_set  |  1 tree(s): salet  |  2,315 paths, 78 tokens, uniform
  step  1: +a    set_before_outside= 0.0000   pairs= 0.0000   top2_diff= 0.0000
  step  2: +e    set_before_outside= 0.0000   pairs= 0.0000   top2_diff= 0.0000
  step  3: +l    set_before_outside= 0.0000   pairs= 0.0000   top2_diff= 0.0000
  step  4: +s    set_before_outside= 0.0000   pairs= 0.0000   top2_diff= 0.0000
  step  5: +t    set_before_outside= 1.8700   pairs= 0.0000   top2_diff= 2.0950

  step  6: +r    set_before_outside= 1.8708   pairs= 1.0950   top2_diff= 0.9490

  step  7: +o    set_before_outside= 1.8730   pairs= 2.7879   top2_diff= 0.9218

  step  8: +n    set_before_outside= 1.8730   pairs= 4.9248   top2_diff= 0.1551
  step  9: +c    set_before_outside= 1.8821   pairs= 6.4756   top2_diff= 0.3857
  step 10: +d    set_before_outside= 1.9965   pairs= 7.8743   top2_diff= 0.2078
  step 11: +i    set_before_outside= 2.0657   pairs= 8.7732   top2_diff= 0.6648

  step 12: +u    set_be

,step,added,set,set_before_outside,pairs_outside_before_set,top2_diff
0,1,a,a,0.0000,0.0000,0.0000
1,2,e,a e,0.0000,0.0000,0.0000
2,3,l,a e l,0.0000,0.0000,0.0000
3,4,s,a e l s,0.0000,0.0000,0.0000
4,5,t,a e l s t,1.8700,0.0000,2.0950
5,6,r,a e l r s t,1.8708,1.0950,0.9490
6,7,o,a e l o r s t,1.8730,2.7879,0.9218
7,8,n,a e l n o r s t,1.8730,4.9248,0.1551
8,9,c,a c e l n o r s t,1.8821,6.4756,0.3857
9,10,d,a c d e l n o r s t,1.9965,7.8743,0.2078


## The resulting order

`ORDER_TOKENS` is the greedy ranking, best first. **The order is computed; the tier CUTS
are an editorial decision** — in hangman the `top2_diff` gaps informed where the rows were
drawn, but the final seven tiers merged some of those groups by hand before being copied
into `animations/assets/tier_list.py`. Same convention here: read the gaps, then choose
the cuts deliberately.

In [6]:
ORDER_TOKENS = list(sel['added'])
print(f'order ({len(D["openers"])} tree(s), {D["nt"]} tokens):')
print('  ' + ' '.join(t.upper() for t in ORDER_TOKENS))
if MAX_REPEAT == 1:
    ORDER_LETTERS = ''.join(ORDER_TOKENS)            # the compact form the tier asset uses
    print(f'  compact: {ORDER_LETTERS.upper()}')

gaps = sel[sel['top2_diff'] > MIN_GAP][['step', 'added', 'top2_diff']]
print(f'\nsteps where top2_diff > {MIN_GAP} (candidate tier boundaries, cut AFTER the step):')
display(gaps)

order (1 tree(s), 78 tokens):
  A E L S T R O N C D I U H B Y P G M K F W EE V TT LL AA OO SS RR Z CC X J Q DD FF II NN MM PP BB GG HH YY KK EEE UU ZZ SSS VV MMM RRR NNN PPP TTT WW DDD FFF BBB JJ QQ XX AAA CCC GGG HHH III JJJ KKK LLL OOO QQQ UUU VVV WWW XXX YYY ZZZ

steps where top2_diff > 0.5 (candidate tier boundaries, cut AFTER the step):


,step,added,top2_diff
4,5,t,2.0950
5,6,r,0.9490
6,7,o,0.9218
10,11,i,0.6648
17,18,m,0.5832
21,22,ee,0.5568


## Which letters are in the openers — and how fast coverage builds

Two different units in one table, so read the column headers carefully:

**Per TREE (the opener alone).** Every path in a tree opens with the same word, so these
count once per tree, not once per path. `occurrences` is the number of times the letter is
typed across all the openers (a `bobby` opener would contribute 3 Bs), `openers` is how
many distinct openers contain it at all, and `pct` is `openers` as a share of the trees
pooled.

**Per PATH (`first1` … `first6`).** These range over **every tree × every answer** — the
first *k* guesses of each of the `NUM_TREES × 2315` paths, cumulative in *k*. Guess 1 is
fixed within a tree, but every guess after it is the reply the answer forces, so unlike the
opener columns these vary with the answer. A path shorter than *k* guesses contributes only
the guesses it has, so the columns flatten out once every path has finished.

`stat` picks what those columns hold:

- `'pct'` *(default)* — share of paths in which the letter has been typed at least once by
  the end of guess *k*. The readable one.
- `'paths'` — the same as a count rather than a share.
- `'occ'` — every copy typed, so a guess with two Es adds 2. Its `first1` column is just the
  opener `occurrences` times 2315.

Neither half depends on `MAX_REPEAT`. The opener columns are the thing to read against the
top of the greedy ranking, since a letter on every opener is tied at rank 0 on every path
and so cannot be ordered against its opener-mates.

In [7]:
def opener_letter_counts(D=None, stat='pct'):
    """Per-letter counts over the pooled trees: the OPENER (once per tree), then coverage
    after the first 1..6 guesses (once per path). `stat` is 'pct', 'paths' or 'occ'."""
    from collections import Counter
    import string
    D = D if D is not None else globals()['D']
    if stat not in ('pct', 'paths', 'occ'):
        raise ValueError(f"stat must be 'pct', 'paths' or 'occ', not {stat!r}")
    openers = [w for w, _total in D['openers']]
    occ, has = Counter(), Counter()
    for w in openers:
        occ.update(w)                       # every typed letter, so doubles count twice
        has.update(set(w))                  # distinct openers carrying the letter
    n, P = len(openers), D['n']
    pre = D['prefix_occ'] if stat == 'occ' else D['prefix_paths']
    K = pre.shape[0]

    rows = []
    for i, c in enumerate(string.ascii_lowercase):
        row = {'letter': c.upper(),
               'occurrences': occ.get(c, 0),
               'openers': has.get(c, 0),
               'pct': round(100.0 * has.get(c, 0) / n, 2) if n else 0.0}
        for k in range(K):
            v = int(pre[k, i])
            row[f'first{k + 1}'] = round(100.0 * v / P, 2) if stat == 'pct' and P else v
        rows.append(row)
    df = pd.DataFrame(rows)
    # rank by how early the letter shows up: first1 first, ties broken by first2, and so on
    by = [f'first{k + 1}' for k in range(K)] + ['letter']
    return df.sort_values(by, ascending=[False] * K + [True]).reset_index(drop=True)


_op = opener_letter_counts()
_tot = D['prefix_occ'].sum(axis=1)
print(f'{len(D["openers"])} opener(s), {int(_op["occurrences"].sum())} letters typed '
      f'({5 * len(D["openers"])} expected = 5 per opener)')
print(f'{D["n"]:,} paths; letters typed by guess k: '
      + ', '.join(f'{k + 1}:{t / D["n"]:.2f}' for k, t in enumerate(_tot))
      + '  (5k is the max, less where a path has already finished)')
print("first1..first6 = % of paths where the letter has been typed by guess k "
      "(opener_letter_counts(stat='paths') or stat='occ' for the raw counts)")
display(_op)

1 opener(s), 5 letters typed (5 expected = 5 per opener)
2,315 paths; letters typed by guess k: 1:5.00, 2:10.00, 3:14.79, 4:16.93, 5:17.11, 6:17.11  (5k is the max, less where a path has already finished)
first1..first6 = % of paths where the letter has been typed by guess k (opener_letter_counts(stat='paths') or stat='occ' for the raw counts)


,letter,occurrences,openers,pct,first1,first2,first3,first4,first5,first6
0,A,1,1,100.0,100.0,100.00,100.00,100.00,100.00,100.00
1,E,1,1,100.0,100.0,100.00,100.00,100.00,100.00,100.00
2,L,1,1,100.0,100.0,100.00,100.00,100.00,100.00,100.00
3,S,1,1,100.0,100.0,100.00,100.00,100.00,100.00,100.00
4,T,1,1,100.0,100.0,100.00,100.00,100.00,100.00,100.00
5,R,0,0,0.0,0.0,77.71,79.83,80.60,80.60,80.60
6,O,0,0,0.0,0.0,58.10,64.84,66.35,66.39,66.39
7,N,0,0,0.0,0.0,44.84,55.03,58.62,58.75,58.75
8,D,0,0,0.0,0.0,38.70,46.22,47.90,47.90,47.90
9,C,0,0,0.0,0.0,38.19,52.35,54.56,54.69,54.69


## Token frequency in the word lists themselves

Share of WORDS containing each token — for a single letter, words holding it at least once
(a word with three Es counts once, not three times); for a repeat token, words holding at
least that many copies. `MAX_REPEAT` sets how far this goes, exactly as it does for the
ranking: `1` gives the 26 letters, `2` adds `EE`, `3` adds `EEE`.

Two lists, straight off disk, with no strategy involved:

- **answers** — `mystery.txt`, the 2315 words that can actually be the answer;
- **guesses** — `guess.txt`, the 12972 legal guesses. Note this list **contains the 2315
  answers**, so the two columns are not independent samples.

This is the baseline the coverage table above should be read against: it says what a token
is worth knowing about, where `first1`…`first6` say when the strategy actually gets to it.
Nothing here depends on `NUM_TREES`.

In [8]:
def letter_word_frequency(max_repeat=None):
    """% of WORDS containing each token, for the answer list and the full guess list.

    A single letter counts a word ONCE however many copies it holds; a repeat token counts
    the words holding at least that many. Both lists are read from the paths module, so
    this is the committed data the solver itself uses."""
    import string
    if max_repeat is None:
        max_repeat = MAX_REPEAT
    lists = {'answer': paths.ANSWER_WORDS, 'guess': paths.FULL_GUESS_WORDS}
    words = {k: [w.strip() for w in open(v) if w.strip()] for k, v in lists.items()}
    rows = []
    for r in range(1, max_repeat + 1):
        for c in string.ascii_lowercase:
            row = {'token': c.upper() * r, 'n': r}
            for k, ws in words.items():
                hits = sum(1 for w in ws if w.count(c) >= r)
                row[f'{k}_words'] = hits
                row[f'{k}_pct'] = round(100.0 * hits / len(ws), 2)
            rows.append(row)
    df = pd.DataFrame(rows).sort_values(['answer_pct', 'token'],
                                        ascending=[False, True]).reset_index(drop=True)
    return df, {k: len(v) for k, v in words.items()}


_wf, _n = letter_word_frequency()
print(f'{_n["answer"]:,} answers (mystery.txt), {_n["guess"]:,} legal guesses (guess.txt) '
      f'-- the guess list CONTAINS the answers')
print('pct = share of WORDS holding the token (a single letter counts a word with 3 Es once; '
      'EE counts words with 2+ Es)')
for r in range(2, MAX_REPEAT + 1):
    _sub = _wf[_wf['n'] == r]
    print(f'  repeat {r}: {int(_sub["answer_words"].sum()):,} answer and '
          f'{int(_sub["guess_words"].sum()):,} guess (token, word) pairs')
display(_wf)

2,315 answers (mystery.txt), 12,972 legal guesses (guess.txt) -- the guess list CONTAINS the answers
pct = share of WORDS holding the letter at least once (a word with 3 Es counts once)


,letter,answer_words,answer_pct,guess_words,guess_pct
0,E,1056,45.62,5705,43.98
1,A,909,39.27,5330,41.09
2,R,837,36.16,3909,30.13
3,O,673,29.07,3911,30.15
4,T,667,28.81,3033,23.38
5,L,648,27.99,3114,24.01
6,I,647,27.95,3589,27.67
7,S,618,26.70,5936,45.76
8,N,550,23.76,2787,21.48
9,U,457,19.74,2436,18.78
